# Tight-Binding Models in Real Space

**Abstract.** This notebook builds tight-binding Hamiltonians on the real-space lattices of the previous notebook. We start from the second-quantised hopping Hamiltonian, discuss how a **hopping map** encodes the model and how Hermiticity is imposed, then use translation symmetry to derive the Bloch Hamiltonian $H(\mathbf k)$ in the **periodic gauge** $H(\mathbf k+\mathbf G)=H(\mathbf k)$. We close with band structures for graphene, kagome, Lieb, dice and the checkerboard Chern insulator, and with the finite real-space Hamiltonian with twisted boundary phases.

**References**

- N. W. Ashcroft and N. D. Mermin, *Solid State Physics* (Saunders College Publishing, 1976), Chs. 8–10.
- D. N. Sheng et al., Phys. Rev. Lett. **107**, 146803 (2011).
- K. Sun, Z. Gu, H. Katsura, and S. Das Sarma, Phys. Rev. Lett. **106**, 236803 (2011).
- T. Fukui, Y. Hatsugai, and H. Suzuki, J. Phys. Soc. Jpn. **74**, 1674 (2005).
- R. Resta, Rev. Mod. Phys. **66**, 899 (1994).

> **Execution note.** This notebook was authored without `nbconvert`, so its cells ship *unexecuted but correct*; the band-structure SVGs referenced below were produced by running the identical code out-of-band with the package venv. Run the cells yourself (from the package root `TightBinding_PY/`, so that `doc/figures/` resolves) to regenerate them.


## 1. Second quantisation and the hopping Hamiltonian

In the tight-binding (LCAO) picture an electron hops between localised orbitals. With $c_i^\dagger$ ($c_i$) the creation (annihilation) operator of an electron at site $i$, the single-particle Hamiltonian is the bilinear form

\begin{equation}
\hat H = \sum_{i,j} t_{ij}\; c_i^\dagger c_j,
\qquad t_{ij}\in\mathbb C .
\end{equation}

Because $\hat H = \hat H^\dagger$, the hopping matrix must be Hermitian,

\begin{equation}
t_{ji} = t_{ij}^* .
\end{equation}

A **diagonal** entry $t_{ii}$ is an on-site (chemical) potential and must be real; an off-diagonal $t_{ij}$ is a hopping amplitude, possibly complex (complex hoppings carry the Peierls phases of a magnetic field or spin-orbit coupling). The package convention: a template is declared for *one* direction, and its Hermitian conjugate is added automatically (`is_hermitian=True`).

A *site* is a `(cell, sublattice)` pair with **1-based** sublattice index, so the sum splits as

\begin{equation}
\hat H = \sum_{\mathbf R,\mathbf R',\alpha,\beta} t_{\alpha\beta}(\mathbf R,\mathbf R')\; c_{\mathbf R,\alpha}^\dagger c_{\mathbf R',\beta},
\end{equation}

and translation invariance reduces $t_{\alpha\beta}(\mathbf R,\mathbf R')$ to a function of the cell difference only, $t_{\alpha\beta}(\mathbf R'-\mathbf R)$.


## 2. Hopping maps: `input_hopping_map` vs `full_hopping_map`

A **hopping map** lists every bond template together with its amplitude. The model object `Real_Space_TightBinding_Model` keeps two complementary maps (both dicts keyed by `((cell, sub), (cell, sub))` with 1-based sub indices):

- `input_hopping_map` — the *minimal* set of templates the user declared (the model definition).
- `full_hopping_map` — those templates *expanded by translation symmetry* over every unit cell, with PBC wrapping (the finite-sample matrix elements).

`add_hopping_term(tb_model, (((cell_from, sub_from), (cell_to, sub_to)), amplitude), is_hermitian=True)` declares one template. Hermiticity is enforced inside: when `is_hermitian=True`, the reverse template is added with the conjugated amplitude. On-site terms use `is_hermitian=False` (so they are not double-counted).


In [ ]:
import numpy as np
from tightbinding_py import *

lat = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 3],
                                    pbc_indicator=[True, True])
tb = initialize_real_space_tightbinding_model(lat, model_name="graphene")

# The three nearest-neighbour bonds of the A1 sublattice.
add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 2)), -1.0))
add_hopping_term(tb, ((((0, 0), 1), ((0, -1), 2)), -1.0))
add_hopping_term(tb, ((((0, 0), 1), ((-1, 0), 2)), -1.0))

print("input_hopping_map keys (with Hermitian conjugates):")
for k, v in tb.input_hopping_map.items():
    print("   ", k, "->", v)
print("\nlen(input_hopping_map) =", len(tb.input_hopping_map),
      "   len(full_hopping_map) =", len(tb.full_hopping_map))


**Hermiticity in practice.** A directed bond `(α, β, Δ)` with amplitude $t$ contributes $t\,c_{\mathbf R,\alpha}^\dagger c_{\mathbf R+\Delta,\beta}$ to $\hat H$; its Hermitian conjugate is the reverse bond `(β, α, −Δ)` with amplitude $t^*$. The three declared $A1\to A2$ templates above produce six `input_hopping_map` entries (three forward, three conjugate), guaranteeing $H(\mathbf k)$ and the real-space $H$ are Hermitian even for complex $t$.


## 3. Translation symmetry and the Bloch expansion

Translation invariance means $t_{\alpha\beta}(\mathbf R,\mathbf R') = t_{\alpha\beta}(\mathbf R'-\mathbf R)$. Relabelling $\mathbf R' = \mathbf R + \Delta$,

\begin{equation}
\hat H = \sum_{\mathbf R,\Delta,\alpha,\beta} t_{\alpha\beta}(\Delta)\; c_{\mathbf R,\alpha}^\dagger c_{\mathbf R+\Delta,\beta},
\qquad t_{\alpha\beta}(\Delta) \equiv t^{0,\alpha;\ \Delta,\beta}.
\end{equation}

Fourier-transform the operators on the $N=\prod_d L_d$ cells (crystal momenta $\mathbf k$ in crystal coordinates, $\mathbf k\cdot\Delta = 2\pi\,\mathbf k_{\mathrm{crys}}\cdot\Delta_{\mathrm{crys}}$):

\begin{equation}
c_{\mathbf R,\alpha} = \frac{1}{\sqrt N}\sum_{\mathbf k} e^{i\mathbf k\cdot\mathbf R}\,c_{\mathbf k,\alpha},
\qquad
c_{\mathbf R,\alpha}^\dagger = \frac{1}{\sqrt N}\sum_{\mathbf k} e^{-i\mathbf k\cdot\mathbf R}\,c_{\mathbf k,\alpha}^\dagger .
\end{equation}

Substituting and using $\sum_{\mathbf R} e^{-i(\mathbf k-\mathbf k')\cdot\mathbf R} = N\,\delta_{\mathbf k\mathbf k'}$,

\begin{align}
\hat H &= \frac1N\sum_{\mathbf R,\Delta,\alpha,\beta}\sum_{\mathbf k,\mathbf k'}
t_{\alpha\beta}(\Delta)\,e^{-i\mathbf k\cdot\mathbf R}\,e^{i\mathbf k'\cdot(\mathbf R+\Delta)}\,c_{\mathbf k,\alpha}^\dagger c_{\mathbf k',\beta}
\\
&= \sum_{\mathbf k}\sum_{\alpha,\beta}\Bigl[\sum_{\Delta}t_{\alpha\beta}(\Delta)\,e^{i\mathbf k\cdot\Delta}\Bigr]c_{\mathbf k,\alpha}^\dagger c_{\mathbf k,\beta}
= \sum_{\mathbf k}\mathbf c_{\mathbf k}^\dagger\,H(\mathbf k)\,\mathbf c_{\mathbf k}.
\end{align}

We thus obtain the **Bloch Hamiltonian**, an $n_{\mathrm{sub}}\times n_{\mathrm{sub}}$ matrix,

\begin{equation}
\boxed{\; H_{\alpha\beta}(\mathbf k) = \sum_{\Delta} t_{\alpha\beta}(\Delta)\,e^{\,i\,2\pi\,\mathbf k\cdot\Delta} \;}
\end{equation}

with $\mathbf k$ and $\Delta$ both in crystal coordinates. The phase depends only on the *cell displacement* $\Delta$, not on the sublattice offsets $\boldsymbol\tau_\alpha$ — a deliberate gauge choice discussed next.


## 4. The periodic gauge

A common alternative writes the phase with the *full site positions*,

\begin{equation}
\widetilde H_{\alpha\beta}(\mathbf k) = \sum_{\Delta} t_{\alpha\beta}(\Delta)\,e^{\,i\,\mathbf k\cdot[(\Delta+\boldsymbol\tau_\beta)-\boldsymbol\tau_\alpha]}.
\end{equation}

This is the gauge natural to the Bloch theorem, but it is **not** periodic in the Brillouin zone: under $\mathbf k\to\mathbf k+\mathbf G$ the extra factor $e^{i\mathbf G\cdot(\boldsymbol\tau_\beta-\boldsymbol\tau_\alpha)}$ is not $1$.

The package instead uses the **periodic gauge**, whose phase $e^{2\pi i\,\mathbf k\cdot\Delta}$ involves only the *integer* cell shift $\Delta$. Then for any reciprocal-lattice vector $\mathbf G$ (integer in crystal coordinates),

\begin{equation}
e^{\,2\pi i\,(\mathbf k+\mathbf G)\cdot\Delta} = e^{\,2\pi i\,\mathbf k\cdot\Delta}
\qquad\Longrightarrow\qquad
H(\mathbf k+\mathbf G) = H(\mathbf k).
\end{equation}

A periodic-in-$\mathbf k$ Hamiltonian has periodic eigenstates (up to a band-dependent phase), the cleanest setting for the Berry-curvature and Chern-number integrals of the next notebook. The two gauges are related by the unitary (but $\mathbf k$-non-periodic) transformation $U_{\alpha\beta}=\delta_{\alpha\beta}e^{-2\pi i\,\mathbf k\cdot\boldsymbol\tau_\alpha}$, so *physical* quantities (spectrum, Chern number) are identical — only the wavefunction phase convention differs.


## 5. Building $H(\mathbf k)$ with `build_Hk_crys`

`build_Hk_crys(tb_model)` returns a closure `Hk_crys(k)` → $n_{\mathrm{sub}}\times n_{\mathrm{sub}}$ matrix in the periodic gauge. It uses `input_hopping_map` when non-empty (the infinite-system templates); otherwise it compresses the graph-generated `full_hopping_map` into one template and divides by $n_{\mathrm{cell}}$.


In [ ]:
Hk_graphene = build_Hk_crys(tb)

# Periodic-gauge check: H(k + G) == H(k) for an integer reciprocal vector G.
k = np.array([0.31, -0.17])
G = np.array([1.0, 1.0])          # integer in crystal coordinates
print("H(k+G) - H(k) max error:", np.max(np.abs(Hk_graphene(k + G) - Hk_graphene(k))))
print("H(0,0) =\n", np.round(Hk_graphene(np.array([0.0, 0.0])), 6))
print("H(0,0) Hermitian error:", np.max(np.abs(Hk_graphene(np.array([0.0, 0.0]))
                                               - Hk_graphene(np.array([0.0, 0.0])).conj().T)))


## 6. Example 1 — graphene (honeycomb lattice)

Graphene has two sublattices with a single nearest-neighbour amplitude $t$. The three $A1\to A2$ bonds point to $A2(\mathbf R)$, $A2(\mathbf R-\mathbf a_1)$, $A2(\mathbf R-\mathbf a_2)$, giving the off-diagonal Hamiltonian

\begin{equation}
H(\mathbf k) = \begin{pmatrix} 0 & t\,f(\mathbf k)\\ t^* f^*(\mathbf k) & 0\end{pmatrix},
\qquad
f(\mathbf k) = 1 + e^{-2\pi i k_1} + e^{-2\pi i k_2},
\end{equation}

with eigenvalues $\varepsilon_\pm = \pm|t|\,|f(\mathbf k)|$. They touch at the Dirac points where $f=0$, namely (in crystal coordinates)

\begin{equation}
\mathbf K = (\tfrac23,\tfrac13),
\qquad
\mathbf K' = (\tfrac13,\tfrac23).
\end{equation}

> **Convention note.** With $\mathbf a_1=(1,0)$, $\mathbf a_2=(1/2,\sqrt3/2)$ and reciprocal basis $\mathbf b_1=2\pi(1,-1/\sqrt3)$, $\mathbf b_2=2\pi(0,2/\sqrt3)$, the Brillouin-zone corner sits at $\mathbf K=(2/3,1/3)$ — *not* $(1/3,1/3)$, which lies inside the zone. The band path below labels $(2/3,1/3)$ as "K".


In [ ]:
from pathlib import Path

FIG_DIR = Path("doc") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

grid = initialize_uniform_grids_from_lattice(lat)
fig, ax = plot_bands(
    Hk_graphene, grid,
    k_path=[[0.0, 0.0], [2/3, 1/3], [0.5, 0.0], [0.0, 0.0]],
    k_path_name_list=["G", "K", "M", "G"],
    nband_range=range(1, 3),
    nk=120,
    save_path=FIG_DIR / "bands_honeycomb.svg",
)
print("band energies at Gamma:", np.linalg.eigvalsh(Hk_graphene(np.array([0.0, 0.0]))))
print("band energies at K    :", np.linalg.eigvalsh(Hk_graphene(np.array([2/3, 1/3]))))


The two bands are degenerate (zero energy) at the $K$ point — the hallmark of graphene's Dirac cones. Opening a gap there (a sublattice potential $M$, or the Haldane complex next-nearest hopping) is the starting point of the topological discussion in the next notebook.


## 7. Example 2 — kagome lattice and its flat band

The kagome lattice has three sublattices, each coupled to four nearest neighbours. Rather than declaring every bond by hand we use `add_hoppings_by_graph_distance(tb, 1, -1.0)`, which attaches a uniform amplitude to **all** pairs of sites at graph distance 1 on the nearest-neighbour graph. With amplitude $-1$ the three-band spectrum consists of two dispersive bands below an **exactly flat** band at $+2$ (the highest band). The flatness is not an accident: the kagome hosts compact localised states — a superposition with alternating signs on the six sites of a hexagon that cancels by destructive interference on every shared site.


In [ ]:
lat_kg = initialize_real_space_lattice(lattice_name="kagome", sample_size=[4, 4],
                                         pbc_indicator=[True, True])
tb_kg = initialize_real_space_tightbinding_model(lat_kg, model_name="kagome")
add_hoppings_by_graph_distance(tb_kg, 1, -1.0)
Hk_kg = build_Hk_crys(tb_kg)

grid_kg = initialize_uniform_grids_from_lattice(lat_kg)
fig, ax = plot_bands(
    Hk_kg, grid_kg,
    k_path=[[0.0, 0.0], [2/3, 1/3], [0.5, 0.0], [0.0, 0.0]],
    k_path_name_list=["G", "K", "M", "G"],
    nband_range=range(1, 4),
    nk=120,
    save_path=FIG_DIR / "bands_kagome.svg",
)

# Confirm the flat band by sampling energies across the whole BZ.
Es = np.array([np.linalg.eigvalsh(Hk_kg(k)) for k in grid_kg.site_crys_list])
print("per-band energy spread over the BZ:", np.round(np.ptp(Es, axis=0), 10))


The flat (third) band has zero spread over the whole Brillouin zone (to machine precision); its energy is $+2$ for the hopping amplitude $-1$ used above.


## 8. Example 3 — Lieb lattice

The Lieb lattice (three sublattices on the square Bravais network) also hosts a flat band from a compact localised state supported on the $A2, A3$ sublattices only. Its three bands are

\begin{equation}
\varepsilon(\mathbf k) = 0
\qquad\text{and}\qquad
\pm |t|\,\sqrt{4 + 2\cos(2\pi k_1) + 2\cos(2\pi k_2)},
\end{equation}

with a flat middle band at $\varepsilon=0$.


In [ ]:
lat_lb = initialize_real_space_lattice(lattice_name="Lieb", sample_size=[5, 5],
                                         pbc_indicator=[True, True])
tb_lb = initialize_real_space_tightbinding_model(lat_lb, model_name="lieb")
add_hoppings_by_graph_distance(tb_lb, 1, -1.0)
Hk_lb = build_Hk_crys(tb_lb)

grid_lb = initialize_uniform_grids_from_lattice(lat_lb)
fig, ax = plot_bands(
    Hk_lb, grid_lb,
    k_path=[[0.0, 0.0], [0.5, 0.0], [0.5, 0.5], [0.0, 0.0]],
    k_path_name_list=["G", "X", "M", "G"],
    nband_range=range(1, 4),
    nk=120,
    save_path=FIG_DIR / "bands_lieb.svg",
)
Es = np.array([np.linalg.eigvalsh(Hk_lb(k)) for k in grid_lb.site_crys_list])
print("per-band energy spread over the BZ:", np.round(np.ptp(Es, axis=0), 10))


## 9. Example 4 — dice (α-T3) lattice

The dice lattice adds a **hub** sublattice at the centre of each honeycomb hexagon: three sites per unit cell, with the hub `A2` connected to three `A1` and three `A3` rim sites (the preset enforces this via `allowed_bonds=[(1,2),(2,3)]`). With uniform nearest-neighbour hopping it also hosts an **exactly flat band** at $\varepsilon=0$, like the Lieb lattice.


In [ ]:
lat_dc = initialize_real_space_lattice(lattice_name="dice", sample_size=[4, 4],
                                         pbc_indicator=[True, True])
tb_dc = initialize_real_space_tightbinding_model(lat_dc, model_name="dice")
add_hoppings_by_graph_distance(tb_dc, 1, -1.0)
Hk_dc = build_Hk_crys(tb_dc)

grid_dc = initialize_uniform_grids_from_lattice(lat_dc)
fig, ax = plot_bands(
    Hk_dc, grid_dc,
    k_path=[[0.0, 0.0], [2/3, 1/3], [0.5, 0.0], [0.0, 0.0]],
    k_path_name_list=["G", "K", "M", "G"],
    nband_range=range(1, 4),
    nk=120,
    save_path=FIG_DIR / "bands_dice.svg",
)
Es = np.array([np.linalg.eigvalsh(Hk_dc(k)) for k in grid_dc.site_crys_list])
print("per-band energy spread over the BZ:", np.round(np.ptp(Es, axis=0), 10))


## 10. Example 5 — checkerboard Chern insulator (Sun–Gu–Katsura–Sarma)

As a bridge to the topology notebooks, we construct the two-orbital checkerboard model of Sun, Gu, Katsura and Das Sarma (PRL **106**, 236803 (2011)). It threads a flux through each square plaquette (complex NN hoppings) and adds real NNN and NNNN hoppings that flatten the lower band. The lattice is built manually (no preset), and the model is declared template-by-template with `add_hopping_term`. Its non-zero Chern number is computed in the next notebook.


In [ ]:
# Checkerboard lattice: square Bravais, two sublattices.
lat_cb = initialize_real_space_lattice(
    brav_vec_list=[[1.0, 0.0], [0.0, 1.0]],
    sample_size=[5, 5],
    sub_crys_list=[[0.5, 0.0], [0.0, 0.5]],
    lattice_name="checkerboard",
    pbc_indicator=[True, True],
)
tb_cb = initialize_real_space_tightbinding_model(lat_cb, model_name="checkerboard_sgks")

t = -1.0
t1_prime = -1.0 / (2.0 + np.sqrt(2.0))
t2_prime = 1.0 / (2.0 + np.sqrt(2.0))
t_double_prime = -1.0 / (2.0 + 2.0 * np.sqrt(2.0))
phi = 2.0 * np.pi / 8.0

# complex nearest-neighbour (flux)
add_hopping_term(tb_cb, ((((0, 0), 1), ((0, 0), 2)), -t * np.exp(-1j * phi)))
add_hopping_term(tb_cb, ((((0, 0), 1), ((1, 0), 2)), -t * np.exp(1j * phi)))
add_hopping_term(tb_cb, ((((0, 0), 2), ((0, 1), 1)), -t * np.exp(-1j * phi)))
add_hopping_term(tb_cb, ((((0, 0), 2), ((-1, 1), 1)), -t * np.exp(1j * phi)))
# real next-nearest-neighbour
add_hopping_term(tb_cb, ((((0, 0), 1), ((1, 0), 1)), -t1_prime))
add_hopping_term(tb_cb, ((((0, 0), 1), ((0, 1), 1)), -t2_prime))
add_hopping_term(tb_cb, ((((0, 0), 2), ((1, 0), 2)), -t2_prime))
add_hopping_term(tb_cb, ((((0, 0), 2), ((0, 1), 2)), -t1_prime))
# real next-next-nearest-neighbour
add_hopping_term(tb_cb, ((((0, 0), 1), ((1, 1), 1)), -t_double_prime))
add_hopping_term(tb_cb, ((((0, 0), 2), ((1, 1), 2)), -t_double_prime))
add_hopping_term(tb_cb, ((((0, 0), 2), ((-1, 1), 2)), -t_double_prime))
add_hopping_term(tb_cb, ((((0, 0), 1), ((1, -1), 1)), -t_double_prime))

Hk_cb = build_Hk_crys(tb_cb)
grid_cb = initialize_uniform_grids_from_lattice(lat_cb)
fig, ax = plot_bands(
    Hk_cb, grid_cb,
    k_path=[[0.0, 0.0], [0.5, 0.0], [0.5, 0.5], [0.0, 0.0]],
    k_path_name_list=["G", "X", "M", "G"],
    nband_range=range(1, 3),
    nk=140,
    save_path=FIG_DIR / "bands_checkerboard.svg",
)

Es = np.array([np.linalg.eigvalsh(Hk_cb(k)) for k in grid_cb.site_crys_list])
print("per-band energy spread:", np.round(np.ptp(Es, axis=0), 6))
print("C_band1 =", Chern_number_Fukui_Hatsugai_Suzuki(Hk_cb, band=1, nk=31))
print("C_band2 =", Chern_number_Fukui_Hatsugai_Suzuki(Hk_cb, band=2, nk=31))


The lower band is nearly flat (its energy spread is much smaller than its gap) and carries Chern number $-1$ — a **flat-band Chern insulator**. The upper band carries $+1$, so the two bands sum to zero as required.

## 11. The finite real-space Hamiltonian (`build_real_space_tb_Hamiltonain`)

For exact diagonalisation and the flux-torus many-body Chern number we need the *finite* $n_{\mathrm{site}}\times n_{\mathrm{site}}$ matrix, not the $n_{\mathrm{sub}}\times n_{\mathrm{sub}}$ Bloch matrix. `build_real_space_tb_Hamiltonain(tb_model, twisted_phases_over_2π=...)` builds it as a sparse `scipy` matrix via `generate_bilinear_terms`: each template is expanded over all cells, bonds leaving open directions are dropped, periodic directions are wrapped, and a hopping crossing direction $d$ with winding $w_d$ acquires the phase

\begin{equation}
t_{ij} \mapsto t_{ij}\,\exp\!\bigl(i\,2\pi\,\theta_d\,w_d\bigr).
\end{equation}


In [ ]:
# Real-space Hamiltonian of graphene on a 2x2 torus.
lat2 = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[2, 2],
                                     pbc_indicator=[True, True])
tb2 = initialize_real_space_tightbinding_model(lat2, model_name="graphene")
add_hopping_term(tb2, ((((0, 0), 1), ((0, 0), 2)), -1.0))
add_hopping_term(tb2, ((((0, 0), 1), ((0, -1), 2)), -1.0))
add_hopping_term(tb2, ((((0, 0), 1), ((-1, 0), 2)), -1.0))

H = build_real_space_tb_Hamiltonain(tb2)            # scipy sparse CSC
Hd = H.toarray()
print("shape:", Hd.shape, " Hermitian error:", np.max(np.abs(Hd - Hd.conj().T)))

terms = generate_bilinear_terms(tb2)
print("n_bilinear_terms:", len(terms), " first:", terms[:3])

# The finite matrix and the k-grid Bloch matrix are the same operator.
Es_finite = np.sort(np.linalg.eigvalsh(Hd))
grid2 = initialize_uniform_grids_from_lattice(lat2)
Es_bloch = np.sort(np.concatenate([np.linalg.eigvalsh(build_Hk_crys(tb2)(k))
                                   for k in grid2.site_crys_list]))
print("max |E_finite - E_bloch| =", np.max(np.abs(Es_finite - Es_bloch)))


**Finite-size remarks.** (i) The finite real-space matrix and the $\mathbf k$-grid Bloch matrix agree *exactly* for a torus of the same size — they are the same operator in Wannier vs Bloch bases. (ii) A flux $\theta_d$ in the real-space matrix is the *same* object as the shifted $\mathbf k$-grid $k_d=(n_d+\theta_d)/L_d$ — this equivalence is what makes the Laughlin pump and the many-body Chern number well defined. (iii) For a band insulator the real-space spectrum develops a gap between filled and empty states that stays open for large enough samples; near a topological phase transition that gap closes and finite-size rounding blurs the transition.
